Check runtime

In [1]:
!nvidia-smi

Sun Feb  8 11:51:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Clone repo

In [3]:
!git clone https://github.com/EnsiyeTahaei/DeepAnT-Time-Series-Anomaly-Detection.git

Cloning into 'DeepAnT-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 56 (delta 6), reused 24 (delta 6), pack-reused 24 (from 3)
Receiving objects: 100% (56/56), 5.36 MiB | 13.28 MiB/s, done.
Resolving deltas: 100% (6/6), done.


Install dependencies

In [ ]:
!pip install -q numpy pandas torch pytorch_lightning scikit-learn matplotlib PyYAML omegaconf

Setup config (bypass argparse which doesn't work in notebooks)

In [6]:
%cd /content/DeepAnT-Time-Series-Anomaly-Detection

/content/DeepAnT-Time-Series-Anomaly-Detection


In [7]:
import os
import random
import logging
import numpy as np
import torch
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load config manually (argparse doesn't work in notebooks)
config = OmegaConf.load("config.yaml")

# Choose your dataset: "air_quality" or "nab"
DATASET = "air_quality"

cfg = OmegaConf.merge(config.common, config.dataset[DATASET])

# Prepare config
os.makedirs(cfg.run_dir, exist_ok=True)
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {cfg.device}")
print(OmegaConf.to_yaml(cfg))

Device: cuda
seed: 200
max_initial_steps: 50
max_steps: 500
batch_size: 16
patience: 150
lr: 0.001
hidden_size: 256
run_dir: ${common.run_dir}/airQuality
dataset_name: AirQuality
window_size: 24
device: cuda



Load data

In [8]:
from utils.data_utils import load_data

train_dataset, val_dataset, test_dataset, feature_dim = load_data(
    cfg.dataset_name, cfg.window_size, cfg.device
)

print(f"Train: {train_dataset.data_x.shape}")
print(f"Val:   {val_dataset.data_x.shape}")
print(f"Test:  {test_dataset.data_x.shape}")
print(f"Features: {feature_dim}")

Train: (7465, 24, 8)
Val:   (933, 24, 8)
Test:  (934, 24, 8)
Features: 8


Train the model

In [ ]:
from deepant.trainer import DeepAnT

model = DeepAnT(cfg, train_dataset, val_dataset, test_dataset, feature_dim)
model.train()

Detect anomalies & visualize

In [ ]:
model.detect_anomaly()

# Display the saved plot inline
from IPython.display import Image, display
display(Image(filename=os.path.join(cfg.run_dir, "anomalies_visualization.png")))